# GAIL

This is an attempt to implement Generative Adversarial Imitation Learning (GAIL) for deterministic policies with off Policy learning on static data. The policy never interacts with the environment (except for evaluation), instead it is trained on policy state-action pair, where policy only selects actions for states sampled from expert data. see [source](https://github.com/nikhilbarhate99/Deterministic-GAIL-PyTorch)

In [68]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt


In [69]:
# Gridworld provides a basic environment for RL agents to interact with

class GridWorld:
    """
    Grid world environment
    """
    
    def __init__(self, length_max, height, width, start_pos):
        """
            input: 
            height - idx : height of the spatial grid
            width - idx : width of the spatial grid
            length - idx : temporal length of a trip
            
            start_pos 2-tuple : coordinates within the state_space (height x width)
            
        """
        self.height = height
        self.width = width
        self.length_max = length_max
        
        self.start = (0, start_pos[0], start_pos[1])
        self.end = (length_max-1, start_pos[0], start_pos[1])
        
        self.n_states = self.height*self.width*self.length_max
        
        self.actions = [0, 1, 2, 3, 4, 5, 6, 7, 8]
        self.n_actions = len(self.actions)
        self.neighbors = [(0, 0),(-1, 0),(-1, 1),(0, 1),(1, 1),(1, 0),(1, -1),(0, -1),(-1, -1)]
        self.dirs = {0: 'stay', 1: 'n', 2: 'ne', 3: 'e', 4: 'se', 5: 's', 6: 'sw', 7: 'w', 8: 'nw'}
    
    def get_grid_idx(self):
        return np.array(range(self.n_states)).reshape((self.length_max, self.height, self.width))
    
    def get_list_state(self):
        return [(i,j,k) for i in range(self.length_max) for j in range(self.height) for k in range(self.width)]
    
    def state2idx(self, state):
        """
        input:
          2d state
        returns:
          1d index
        """
        return self.get_grid_idx()[state]

    def idx2state(self, idx):
        """
        input:
          1d idx
        returns:
          2d state
        """
        return self.get_list_state()[idx]
           
    def get_next_state(self, state, action):
        """
        get next state with [action] on [state]
        args
          state     (z, y, x)
          action    int
        returns
          new state
        """
        if state[0] >= self.length_max-1:
            return state
        else :
            inc = self.neighbors[action]
            nei_s = (state[1] + inc[0], state[2] + inc[1])
            if nei_s[0] >= 0 and nei_s[0] < self.height and nei_s[1] >= 0 and nei_s[1] < self.width:
                next_state = (state[0] + 1, nei_s[0], nei_s[1])
            else:
                next_state = (state[0] + 1, state[1], state[2])
            return next_state

    def get_list_previous_state(self, state):
        """
        args
          state     (z, y, x)
        returns
          tuple
              - previous state (z, y, x)
              - associated action int
        """
        previous = []
        for a in self.actions:
            inc = self.neighbors[a]
            nei_s = (state[1] - inc[0], state[2] - inc[1])

            if nei_s[0] >= 0 and nei_s[0] < self.height and nei_s[1] >= 0 and nei_s[1] < self.width:
                previous_state = (state[0] - 1, nei_s[0], nei_s[1])
                previous.append((previous_state,a))
        return previous

    def get_transition_mat(self):
        """
        get transition dynamics of the gridworld
        return:
          P_a         NxNxN_ACTIONS transition probabilities matrix - 
                        P_a[s0, s1, a] is the transition prob of 
                        landing at state s1 when taking action 
                        a at state s0
        """
        P_a = np.zeros((self.n_states, self.n_states, self.n_actions))
        
        for i in range(self.n_states):
            si = self.idx2state(i)
            for a in range(self.n_actions):
                sj = self.get_next_state(si,a)
                j = self.state2idx(sj)
                P_a[i, j, a] = 1                
        return P_a

In [70]:
g = GridWorld(5,3,4,(1,1))

In [71]:
g.get_grid_idx()

array([[[ 0,  1,  2,  3],
        [ 4,  5,  6,  7],
        [ 8,  9, 10, 11]],

       [[12, 13, 14, 15],
        [16, 17, 18, 19],
        [20, 21, 22, 23]],

       [[24, 25, 26, 27],
        [28, 29, 30, 31],
        [32, 33, 34, 35]],

       [[36, 37, 38, 39],
        [40, 41, 42, 43],
        [44, 45, 46, 47]],

       [[48, 49, 50, 51],
        [52, 53, 54, 55],
        [56, 57, 58, 59]]])

In [193]:
## feat_map
map_mask = [[-1, -1, -1, 0],
            [-1, 0, 0, 0],
            [-1, 0, 0, 0]]

map_dist = [[-0.5, -0.5, -0.5, -1],
            [-0.5, 0, -0.5, -1],
            [-0.5, -0.5, -0.5, -1]]

map_gradv = [[0, 0, 0, 0],
            [-0.5, -0.5, -0.5, -0.5],
            [-1, -1, -1, -1]]

map_gradh = [[0, -0.3, -0.6, -0.9],
            [0, -0.3, -0.6, -0.9],
            [0, -0.3, -0.6, -0.9]]

map_const = [[0, 0, 0, 0],
            [0, 0, 0, 0],
            [0, 0, 0, 0]]

feat_map = np.array([map_mask, map_dist, map_gradv, map_gradh, map_const]).reshape(5,12).T


## Create expert trajectories

trajs = [[[0,1,1], [3]], [[1,1,2], [5]], [[2,2,2], [7]], [[3,2,1], [2]], [[4,1,2], [7]], [[5,1,1], [0]]]

In [194]:
np.random.randint(0, len(trajs), 2)

array([2, 2])

In [195]:
class ExpertTraj:
    def __init__(self, trajs):
        
        self.trajs = trajs
        self.exp_states = [t[0] for t in self.trajs]
        self.exp_actions = [t[1] for t in self.trajs]
        
    def sample(self, batch_size):
        idx = np.random.randint(0, len(self.trajs), batch_size)
        state, action = [], []
        for i in idx:
            s = self.exp_states[i]
            a = self.exp_actions[i]
            state.append(np.array(s, copy=False))
            action.append(np.array(a, copy=False))

        return np.array(state), np.array(action)

In [196]:
# class ExpertTraj:
#     def __init__(self, env_name):
#         self.exp_states = np.loadtxt("./expert_traj/{}/{}_expert_states.dat".format(env_name, env_name))
#         self.exp_actions = np.loadtxt("./expert_traj/{}/{}_expert_actions.dat".format(env_name, env_name))
#         self.n_transitions = len(self.exp_actions)
    
#     def sample(self, batch_size):
#         indexes = np.random.randint(0, self.n_transitions, size=batch_size)
#         state, action = [], []
#         for i in indexes:
#             s = self.exp_states[i]
#             a = self.exp_actions[i]
#             state.append(np.array(s, copy=False))
#             action.append(np.array(a, copy=False))
#         return np.array(state), np.array(action)

In [197]:
expert = ExpertTraj(trajs)

In [198]:
expert.sample(2)

(array([[2, 2, 2],
        [4, 1, 2]]), array([[7],
        [7]]))

In [78]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

# Actor NN 

Estimate Action from state

In [163]:
class Actor(nn.Module):
    def __init__(self, state_dim, action_dim, max_action):
        super(Actor, self).__init__()
        
        self.l1 = nn.Linear(state_dim, 400)
        self.l2 = nn.Linear(400, 300)
        self.l3 = nn.Linear(300, action_dim)
        
        self.max_action = max_action
        
    def forward(self, x):
        x = F.relu(self.l1(x))
        x = F.relu(self.l2(x))
        x = F.relu(self.l3(x)) * self.max_action
        return x

In [164]:
g.actions

[0, 1, 2, 3, 4, 5, 6, 7, 8]

In [165]:
state_dim = 3
action_dim = 1
max_action = 9

a = Actor(state_dim, action_dim, max_action)

In [166]:
state = torch.FloatTensor( [[0,1,1]]).to(device)

In [167]:
a(torch.tensor(state))

/home/amdroy/anaconda3/lib/python3.7/site-packages/ipykernel_launcher.py:1: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  """Entry point for launching an IPython kernel.


tensor([[0.]], grad_fn=<MulBackward0>)

# Discriminator NN

Estimate probability from action, state

In [168]:
class Discriminator(nn.Module):
    def __init__(self, state_dim, action_dim):
        super(Discriminator, self).__init__()
        
        self.l1 = nn.Linear(state_dim+action_dim, 40)
        self.l2 = nn.Linear(40, 30)
        self.l3 = nn.Linear(30, 1)
    
    def forward(self, state, action):
        state_action = torch.cat([state, action], 1)
        x = torch.tanh(self.l1(state_action))
        x = torch.tanh(self.l2(x))
        x = torch.sigmoid(self.l3(x))
        return x

In [169]:
d = Discriminator(state_dim, action_dim)

In [170]:
state = torch.FloatTensor( [[0,1,1]]).to(device)
action = torch.FloatTensor( [[1]]).to(device)

In [171]:
state

tensor([[0., 1., 1.]])

In [172]:
d(state, action)

tensor([[0.5530]], grad_fn=<SigmoidBackward>)

In [182]:
class GAIL:
    def __init__(self, trajs, state_dim, action_dim, max_action, lr):
        self.actor = Actor(state_dim, action_dim, max_action).to(device)
        self.optim_actor = torch.optim.Adam(self.actor.parameters(), lr=lr)
        
        self.discriminator = Discriminator(state_dim, action_dim).to(device)
        self.optim_discriminator = torch.optim.Adam(self.discriminator.parameters(), lr=lr)
        
        self.max_action = max_action
        self.expert = trajs
        
        self.loss_fn = nn.BCELoss()
    
    def select_action(self, state):
        state = torch.FloatTensor(state).to(device)
        return self.actor(state).cpu().data.numpy().flatten()
        
    def update(self, n_iter, batch_size=2):
        for i in range(n_iter):
            # sample expert transitions
            exp_state, exp_action = self.expert.sample(batch_size)
            exp_state = torch.FloatTensor(exp_state).to(device)
            exp_action = torch.FloatTensor(exp_action).to(device)
            
            # sample expert states for actor
            state, _ = self.expert.sample(batch_size)
            state = torch.FloatTensor(state).to(device)
            action = self.actor(state)
            
            #######################
            # update discriminator
            #######################
            self.optim_discriminator.zero_grad()
            
            # label tensors
            exp_label= torch.full((batch_size,1), 1, dtype = torch.float32, device=device)
            policy_label = torch.full((batch_size,1), 0, dtype = torch.float32, device=device)
            
            # with expert transitions
            prob_exp = self.discriminator(exp_state, exp_action)
            loss = self.loss_fn(prob_exp, exp_label)
            
            # with policy transitions
            prob_policy = self.discriminator(state, action.detach())
            loss += self.loss_fn(prob_policy, policy_label)
            
            # take gradient step
            loss.backward()
            self.optim_discriminator.step()
            
            ################
            # update policy
            ################
            self.optim_actor.zero_grad()
            
            loss_actor = -self.discriminator(state, action)
            loss_actor.mean().backward()
            self.optim_actor.step()
            

In [183]:
policy = GAIL(expert, 3, 1, 1, 0.05)

In [184]:
policy.select_action(state)

array([0.08409915], dtype=float32)

In [185]:
policy.update(10, batch_size=1)

In [186]:
######### Hyperparameters #########
random_seed = 0
max_timesteps = 5        # max time steps in one episode
n_eval_episodes = 20        # evaluate average reward over n episodes
lr = 0.0002                 # learning rate
n_epochs = 200              # number of epochs
n_iter = 100                # updates per epoch
batch_size = 100            # num of transitions sampled from expert
###################################
    

In [187]:
action

tensor([[1.]])

In [188]:
policy.select_action(g.start)

array([0.], dtype=float32)

In [189]:
import IPython

In [190]:
policy = GAIL(expert, 3, 1, 1, 0.05)

# training procedure
for epoch in range(1, n_epochs+1):
    
    print("Epoch: " + str(epoch) + "/" + str(n_epochs))
    IPython.display.clear_output(wait=True)
    
    # update policy n_iter times
    policy.update(n_iter, batch_size)

    # evaluate in environment
    total_reward = 0
    for episode in range(n_eval_episodes):
        state = g.start
        for t in range(max_timesteps):
            action = int(policy.select_action(state))
            state = g.get_next_state(state, action)
    

Epoch: 200/200


In [191]:
simulations = []

for episode in range(n_eval_episodes):
    traj = []
    state = g.start
    traj.append(state)
    for t in range(max_timesteps):
        action = int(policy.select_action(state))
        state = g.get_next_state(state, action)
        traj.append(state)
    
    simulations.append(traj)
    

In [192]:
simulations

[[(0, 1, 1), (1, 1, 1), (2, 1, 1), (3, 1, 1), (4, 1, 1), (4, 1, 1)],
 [(0, 1, 1), (1, 1, 1), (2, 1, 1), (3, 1, 1), (4, 1, 1), (4, 1, 1)],
 [(0, 1, 1), (1, 1, 1), (2, 1, 1), (3, 1, 1), (4, 1, 1), (4, 1, 1)],
 [(0, 1, 1), (1, 1, 1), (2, 1, 1), (3, 1, 1), (4, 1, 1), (4, 1, 1)],
 [(0, 1, 1), (1, 1, 1), (2, 1, 1), (3, 1, 1), (4, 1, 1), (4, 1, 1)],
 [(0, 1, 1), (1, 1, 1), (2, 1, 1), (3, 1, 1), (4, 1, 1), (4, 1, 1)],
 [(0, 1, 1), (1, 1, 1), (2, 1, 1), (3, 1, 1), (4, 1, 1), (4, 1, 1)],
 [(0, 1, 1), (1, 1, 1), (2, 1, 1), (3, 1, 1), (4, 1, 1), (4, 1, 1)],
 [(0, 1, 1), (1, 1, 1), (2, 1, 1), (3, 1, 1), (4, 1, 1), (4, 1, 1)],
 [(0, 1, 1), (1, 1, 1), (2, 1, 1), (3, 1, 1), (4, 1, 1), (4, 1, 1)],
 [(0, 1, 1), (1, 1, 1), (2, 1, 1), (3, 1, 1), (4, 1, 1), (4, 1, 1)],
 [(0, 1, 1), (1, 1, 1), (2, 1, 1), (3, 1, 1), (4, 1, 1), (4, 1, 1)],
 [(0, 1, 1), (1, 1, 1), (2, 1, 1), (3, 1, 1), (4, 1, 1), (4, 1, 1)],
 [(0, 1, 1), (1, 1, 1), (2, 1, 1), (3, 1, 1), (4, 1, 1), (4, 1, 1)],
 [(0, 1, 1), (1, 1, 1), (2, 1, 1),